# Writing a database

In [notebook 01](01-first-footprint.ipynb) you opened a database that was already written. Here you write one yourself, one file at a time, in a temporary folder. The database holds one tote bag. It buys 0.15 kg of cotton fabric and 0.1 kWh of electricity from the toy background of notebook 01.

The notebook has four parts:

1. **The four files every database has**: `database.yaml`, `flows.csv`, and one pair of files per process. Then a first footprint.
2. **The optional files**, one by one, each with a footprint you can check by hand: `overrides/`, `parameters.yaml`, `cut_offs.csv`, `no_factor.csv`, and `uses`, which lets a database buy from another database.
3. **A study**: a folder next to the database that asks it a question, with products, amounts and scenarios.
4. **When something is wrong**: common mistakes, and the message headwater gives for each.

In Brightway you would build this in Python, or in an Excel sheet, and write it into a project. In headwater a database is a folder of text files you keep in git. The formats are in [docs/design.md](../docs/design.md#a-database), and the terms in the [glossary](../docs/appendix/glossary.md).

## Set up

This cell loads the toy background, as in notebook 01, and makes a temporary folder for the database. It also defines three small helpers that are not part of headwater:

- `write(name, text)` writes one file of the database.
- `show(name)` prints one file of the database.
- `expect_error(folder, code, step)` opens a database and calculates its tote bag, `tote:tote`, which must fail. It prints the message, and checks which error came back and from which step. If a mistake ever stops being an error, CI fails.

In [1]:
import difflib
import math
import shutil
import tempfile
from contextlib import contextmanager
from pathlib import Path

import headwater as hw

here = Path.cwd()
toy = next(
    (
        p / "tests/fixtures/toy"
        for p in [here, *here.parents]
        if (p / "tests/fixtures/toy").is_dir()
    ),
    None,
)
assert toy, "tests/fixtures/toy not found: run this notebook from a checkout of the repository"
bg = hw.load(toy / "background", method=toy / "methods/toy-climate", namespace="toy")

workdir = tempfile.TemporaryDirectory()
databases = Path(workdir.name)
database_folder = databases / "tote"


def write(name, text, folder=database_folder):
    """Write one file of a database, and make its folder if needed."""
    path = folder / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text)


def show(name, folder=database_folder):
    print(f"{name}:")
    print((folder / name).read_text())


def expect_error(folder, code, step):
    """Open a database and calculate its tote bag, which must fail with the problem `code`,
    raised in `step`."""
    place = "hw.open"
    try:
        db = hw.open(folder)
        place = "db.calc"
        db.calc(bg, "tote:tote")
    except hw.InvalidDataError as error:
        print(f"Stopped in {place}:")
        print(error)
        codes = [problem.code for problem in error.problems]
        assert code in codes, f"expected {code!r}, got {codes}"
        assert place == step, f"expected the error in {step}, but it came from {place}"
        return
    raise AssertionError("the database should have failed, but it calculated")

# Part 1: the four files every database has

```
tote/
  database.yaml          its name, its background and its methods
  flows.csv              the flows the database defines itself
  processes/tote.yaml    the name, location and kind of one process
  processes/tote.csv     the lines of that process
```

## `database.yaml`

It says what the database is and what it is scored on:

- `name` is the start of the database's process ids: our process will be `tote:tote`. Keep it the folder's name: another database finds it by that name.
- `background` names the release the database was written for, as `name@hash`. A database that pins a release is refused on any other. The toy background is a folder, not a release, so it gets a label: `toy-background@fixture`.
- `methods` lists the methods it is scored with, here `[toy-climate]`. It is optional. When it is given, a background loaded with other methods is refused.
- `description`, `allocation` (how a process that makes several products is split) and `end_of_life` are optional text for people. No calculation reads them.

One more field, `cut_off`, comes with the cut-offs in Part 2.

Every process of a database is a product, scored per unit of what it makes: `tote:tote` per tote bag. So there is no goal, functional unit or system boundary here: those belong to a study, in Part 3.

In [2]:
write(
    "database.yaml",
    """\
name: tote
background: toy-background@fixture
methods: [toy-climate]
description: One tote bag, a teaching example
""",
)

## `flows.csv`

A database defines the flows of its own products here: the tote bag. Each flow has an `id`, a `name`, a `kind` (`product`, `waste` or `elementary`) and a reference `unit`. Flows of the background, such as `toy:cotton-fabric`, are not listed again.

In [3]:
write(
    "flows.csv",
    """\
id,name,kind,unit
tote:tote-bag,tote bag,product,unit
""",
)

## `processes/tote.yaml`

A **process** is a pair of files with the same name. The `.yaml` holds its `name`, its `location` (an ecoinvent geography code: `GLO` is global) and its `kind`, the ecoSpold2 activity type. A factory is an `ordinary_transforming_activity`. The process id is the database's name and the file name: `tote:tote`.

In [4]:
write(
    "processes/tote.yaml",
    """\
name: tote bag
location: GLO
kind: ordinary_transforming_activity
""",
)

## `processes/tote.csv`

The `.csv` holds the process's lines, one per row, sorted by line id. A line is what Brightway calls an exchange. Its `role` says what it means:

| Role | Meaning | Brightway exchange type |
|---|---|---|
| `reference` | what the process makes, one per process | production |
| `input` | something bought, from the process named in `supplier` | technosphere |
| `emission` | something released to nature, with no supplier | biosphere |
| `waste_to_treatment` | waste sent to a treatment, in kg sent | technosphere, with a negative amount |

The other roles (`own_use`, `recyclable`, `avoided_product`, `resource`) are in [docs/design.md](../docs/design.md#roles-and-signs). Amounts are positive, and the role decides the sign in the calculation. `supplier` is the id of the process that makes the input; [notebook 00](00-getting-started.ipynb) shows how `bg.find` gives it.

In [5]:
write(
    "processes/tote.csv",
    """\
line,role,flow,amount,unit,supplier
e01,reference,tote:tote-bag,1,unit,
e02,input,toy:cotton-fabric,0.15,kg,toy:fabric-production
e03,input,toy:electricity,0.1,kWh,toy:electricity-production
""",
)

for path in sorted(database_folder.rglob("*")):
    if path.is_file():
        print(path.relative_to(database_folder))

database.yaml
flows.csv
processes/tote.csv
processes/tote.yaml


## Calculate

`hw.open` reads the folder and checks each file. `db.validate(bg)` checks it against the background without calculating, and gives a report of errors and warnings. `db.calc(bg, "tote:tote")` calculates one unit of what the process `tote:tote` makes: one tote bag.

By hand: 0.15 kg × 3.7086 + 0.1 kWh × 0.5298 = 0.55629 + 0.05298 = 0.60927 kg CO₂-eq.

In [6]:
db = hw.open(database_folder)
print(db.validate(bg))
result = db.calc(bg, "tote:tote")
print(result)
assert math.isclose(result.score, 0.60927, rel_tol=1e-9)

Report(no problems)
Result(tote:tote = 0.60927 kg CO2-eq, indicator climate-change, method toy-climate 1)


The report is empty: no errors, no warnings. The tote bag is 0.60927 kg CO₂-eq.

# Part 2: the optional files

Each section adds one file, shows it, and calculates. The numbers are checked by hand.

## `overrides/`: change a background process, for this database only

Say the tote factory has a wind power contract: its electricity emits 0.3 kg CO₂ per kWh, not 0.5. That is a change to a background process, `toy:electricity-production`, that holds only for what this database buys.

An **override** is a CSV file in `overrides/`, any name. Each row is a whole line of the background process, as it should be for this database. `role` and `flow` must be those of the background line, so a reviewer sees what the line is.

In [7]:
print((toy / "background/processes/electricity-production.csv").read_text())
write(
    "overrides/wind-power.csv",
    """\
process,line,role,flow,amount,unit,supplier,comment
toy:electricity-production,e02,emission,toy:co2,0.3,kg,,Wind power contract: 0.3 kg CO2 per kWh
""",
)

line,role,flow,amount,unit,supplier
e01,reference,toy:electricity,1,kWh,
e02,emission,toy:co2,0.5,kg,
e03,emission,toy:ch4,0.001,kg,



The database now gets its own **copy** of the electricity process, with its own id, `tote:copy:toy:electricity-production`. Only the database's own purchases use the copy. The fabric still buys the grid's electricity, and the background is not changed.

By hand: the copy scores 0.3 + 0.001 × 29.8 = 0.3298 per kWh, so the tote is 0.55629 + 0.1 × 0.3298 = 0.58927. The tree marks the copy.

In [8]:
result = hw.open(database_folder).calc(bg, "tote:tote")
print(result)
print(result.contributions())
assert math.isclose(result.score, 0.58927, rel_tol=1e-9)

Result(tote:tote = 0.58927 kg CO2-eq, indicator climate-change, method toy-climate 1)
1 unit of tote bag: 0.58927 (100.0%), direct 0.00000
  0.15 kg of cotton fabric production: 0.55629 (94.4%), direct 0.31788
    0.45 kWh of electricity production: 0.23841 (40.5%), direct 0.23841
  0.1 kWh of electricity production (local copy): 0.03298 (5.6%), direct 0.03298


The next sections start from the tote without the override, so remove the file.

In [9]:
(database_folder / "overrides/wind-power.csv").unlink()
(database_folder / "overrides").rmdir()
assert math.isclose(hw.open(database_folder).calc(bg, "tote:tote").score, 0.60927, rel_tol=1e-9)

## `parameters.yaml`: named values and formulas

Instead of writing 0.15 in the line, give the number a name. `parameters.yaml` holds named values, and formulas of other names. Here the fabric comes from the bag's area and the fabric's weight per m²: 0.5 m² × 0.3 kg/m² = 0.15 kg. An entry is a number, a formula, or a mapping with `value` or `formula` and optionally `unit` and `comment`. A unit must be one headwater knows, such as `kg` or `m2`. The formula syntax is close to Python and to Brightway's Excel formulas ([docs/design.md](../docs/design.md#parameters-and-formulas)).

Line `e02` then says `fabric_kg` instead of 0.15. The cell shows the change to the line as a diff.

In [10]:
write(
    "parameters.yaml",
    """\
database:
  bag_area_m2:
    value: 0.5
    unit: m2
    comment: Fabric area of one bag, from the pattern
  fabric_kg_per_m2:
    value: 0.3
    comment: Weight of the fabric, in kg per m2
  fabric_kg:
    formula: bag_area_m2 * fabric_kg_per_m2
    unit: kg
""",
)
lines = database_folder / "processes/tote.csv"
before = lines.read_text()
lines.write_text(before.replace(",0.15,", ",fabric_kg,"))
diff = difflib.unified_diff(before.splitlines(), lines.read_text().splitlines(), lineterm="", n=0)
print("\n".join(list(diff)[2:]))

@@ -3 +3 @@
-e02,input,toy:cotton-fabric,0.15,kg,toy:fabric-production
+e02,input,toy:cotton-fabric,fabric_kg,kg,toy:fabric-production


The score does not change: 0.60927. What changes is that `result.parameters()` now ranks the named parameters, as `sensitivity` ranks lines. Each one's elasticity is the % change of the score per 1% change of the parameter. All three are the same here, because the fabric is the product of the two: (0.15 × 3.7086) / 0.60927 = 0.9130.

In [11]:
result = hw.open(database_folder).calc(bg, "tote:tote")
print(result)
for parameter in result.parameters():
    formula = f" = {parameter.formula}" if parameter.formula else ""
    unit = f" {parameter.unit}" if parameter.unit else ""
    print(
        f"{parameter.parameter_id}{formula}: {parameter.value:g}{unit}, "
        f"sensitivity {parameter.sensitivity:.5f}, elasticity {parameter.elasticity:.4f}"
    )
assert math.isclose(result.score, 0.60927, rel_tol=1e-9)
expected = 0.15 * 3.7086 / 0.60927
assert all(math.isclose(parameter.elasticity, expected) for parameter in result.parameters())

Result(tote:tote = 0.60927 kg CO2-eq, indicator climate-change, method toy-climate 1)
bag_area_m2: 0.5 m2, sensitivity 1.11258, elasticity 0.9130
fabric_kg = bag_area_m2 * fabric_kg_per_m2: 0.15 kg, sensitivity 3.70860, elasticity 0.9130
fabric_kg_per_m2: 0.3, sensitivity 1.85430, elasticity 0.9130


## `cut_offs.csv`: leave a line out on purpose

The tote is sewn with 0.002 kg of thread, and the background has no thread. A line with no supplier is an error, so a missing input is never counted as 0 by accident. Add the thread to `flows.csv` and as line `e04`, with no supplier, and see:

In [12]:
with (database_folder / "flows.csv").open("a") as file:
    file.write("tote:thread,sewing thread,product,kg\n")
with lines.open("a") as file:
    file.write("e04,input,tote:thread,0.002,kg,\n")
expect_error(database_folder, "supplier-missing", "db.calc")

Stopped in db.calc:
1 problem:
- tote:tote line e04: input of flow 'tote:thread' has no supplier. Name the process that makes it. To leave it out on purpose, record a cut-off in the database's cut_offs.csv.


To leave it out on purpose, record a **cut-off**: one row of `cut_offs.csv` with the line, the criterion, its share, the reason, who decided and when. The share is your estimate: 0.002 kg of 0.152 kg of material is 1.32% of the mass.

In [13]:
write(
    "cut_offs.csv",
    """\
process,line,flow,criterion,share,reason,who,when
tote:tote,e04,tote:thread,mass,0.0132,No thread in the background: 0.002 of 0.152 kg,leo,2026-10-20
""",
)
expect_error(database_folder, "cut-off-no-limit", "hw.open")

Stopped in hw.open:
1 problem:
- cut_offs.csv: has 1 row, and database.yaml has no cut_off to check them against. Add the criteria and the largest share that may be left out in all, such as `cut_off: {criteria: [mass, energy], max_cumulative: 0.05}`. [cut_offs.csv]


The row alone is refused: a database with cut-off rows must also say how much may be left out in all. That is `cut_off` in `database.yaml`: the criteria, and `max_cumulative`, the largest share in all, 3% here. headwater adds up the shares by criterion and checks them against it. A database with no cut-off rows needs no `cut_off`.

In [14]:
with (database_folder / "database.yaml").open("a") as file:
    file.write("cut_off: {criteria: [mass], max_cumulative: 0.03}\n")
result = hw.open(database_folder).calc(bg, "tote:tote")
print(result)
print(
    "cut off:",
    [(issue.process_id, issue.line_id, issue.reason) for issue in result.provenance.cut_offs],
)
print("shares: ", {c.value: share for c, share in result.provenance.cut_off_shares.items()})

assert math.isclose(result.score, 0.60927, rel_tol=1e-9)

Result(tote:tote = 0.60927 kg CO2-eq, indicator climate-change, method toy-climate 1)
cut off: [('tote:tote', 'e04', 'No thread in the background: 0.002 of 0.152 kg')]
shares:  {'mass': 0.0132}


The line buys nothing, so the score stays 0.60927, and the result's provenance lists the cut-off with its reason. A share above 3% in all would be refused, naming the lines that add up.

## `no_factor.csv`: a flow the method does not count

Say the tote factory emits 0.001 kg of nitrous oxide. The toy method has factors for CO₂ and methane only, so this flow would count as 0. Brightway counts it 0 and says nothing. headwater refuses the database until a person has checked that 0 is right:

In [15]:
with (database_folder / "flows.csv").open("a") as file:
    file.write("tote:n2o,Dinitrogen monoxide,elementary,kg\n")
with lines.open("a") as file:
    file.write("e05,emission,tote:n2o,0.001,kg,\n")
expect_error(database_folder, "no-factor", "db.calc")

Stopped in db.calc:
1 problem:
- flow tote:n2o: has no factor in method toy-climate; used by tote:tote. To count it as 0, add a row to the database's no_factor.csv: flow, method, reason, who, when.


The fix is a factor in the method, or one row of `no_factor.csv` that says the flow counts 0, why, who and when. Then the database calculates, and `result.not_characterised` lists the flow with that reason, so the 0 is never silent.

Flows of the **background** that the method does not cover need no row: ecoinvent has thousands. They are listed on the result too.

In [16]:
write(
    "no_factor.csv",
    """\
flow,flow_name,method,reason,who,when
tote:n2o,Dinitrogen monoxide,toy-climate,toy-climate has no N2O factor: 0 here,leo,2026-10-20
""",
)
result = hw.open(database_folder).calc(bg, "tote:tote")
print(result)
for flow in result.not_characterised:
    own = flow.in_database
    print(f"{flow.flow_id} ({flow.name}): in the database {own}, used by {flow.processes}")
    print(f"  reason: {flow.reason}")

assert math.isclose(result.score, 0.60927, rel_tol=1e-9)

Result(tote:tote = 0.60927 kg CO2-eq, indicator climate-change, method toy-climate 1, 1 flow not characterised)
tote:n2o (Dinitrogen monoxide): in the database True, used by ('tote:tote',)
  reason: toy-climate has no N2O factor: 0 here


## `uses`: buy from another database

Many databases share processes: dyes, chemicals, packaging. Write them once, as a database of their own, in a folder next to the others. A database that buys from it lists it in `uses:`, and its lines name the other database's process ids.

The toy has a `dyes` database: 1 kg of reactive dye takes 2 kWh and emits 4 kg CO₂, so it scores 2 × 0.5298 + 4 = 5.0596 per kg. Copy it next to the tote:

In [17]:
shutil.copytree(toy / "tshirt-dyed/dyes", databases / "dyes")
show("database.yaml", databases / "dyes")
show("processes/reactive-dye.csv", databases / "dyes")

database.yaml:
name: dyes
background: toy-background@fixture
description: Dyes that the toy T-shirts buy, written once

processes/reactive-dye.csv:
line,role,flow,amount,unit,supplier
e01,reference,dyes:reactive-dye,1,kg,
e02,input,toy:electricity,kwh_per_kg,kWh,toy:electricity-production
e03,emission,toy:co2,4,kg,



Its `database.yaml` lists no methods: they are optional. The tote buys 0.01 kg of dye. Add `uses: [dyes]` to the tote's `database.yaml`, and a line that buys `dyes:reactive-dye`. By hand: 0.60927 + 0.01 × 5.0596 = 0.659866.

In [18]:
with (database_folder / "database.yaml").open("a") as file:
    file.write("uses: [dyes]\n")
with lines.open("a") as file:
    file.write("e06,input,dyes:reactive-dye,0.01,kg,dyes:reactive-dye\n")
result = hw.open(database_folder).calc(bg, "tote:tote")
print(result)
print(result.contributions())
print("databases used:", result.provenance.used)
assert math.isclose(result.score, 0.659866, rel_tol=1e-9)

Result(tote:tote = 0.659866 kg CO2-eq, indicator climate-change, method toy-climate 1, 1 flow not characterised)
1 unit of tote bag: 0.65987 (100.0%), direct 0.00000
  0.15 kg of cotton fabric production: 0.55629 (84.3%), direct 0.31788
    0.45 kWh of electricity production: 0.23841 (36.1%), direct 0.23841
  0.1 kWh of electricity production: 0.05298 (8.0%), direct 0.05298
  0.01 kg of reactive dye production (dyes:reactive-dye): 0.05060 (7.7%), direct 0.04000
    rest: 0.01060 (1.6%)
databases used: ('dyes',)


The dye shows with its full id, so you see which database it comes from. There is no version to pin: git versions both databases, and a change to `dyes` shows in the results diff of every database that uses it. All databases of a chain must pin the same background.

Here is the whole database now:

In [19]:
for path in sorted(database_folder.rglob("*")):
    if path.is_file():
        print(path.relative_to(database_folder))
print()
show("processes/tote.csv")

cut_offs.csv
database.yaml
flows.csv
no_factor.csv
parameters.yaml
processes/tote.csv
processes/tote.yaml

processes/tote.csv:
line,role,flow,amount,unit,supplier
e01,reference,tote:tote-bag,1,unit,
e02,input,toy:cotton-fabric,fabric_kg,kg,toy:fabric-production
e03,input,toy:electricity,0.1,kWh,toy:electricity-production
e04,input,tote:thread,0.002,kg,
e05,emission,tote:n2o,0.001,kg,
e06,input,dyes:reactive-dye,0.01,kg,dyes:reactive-dye



# Part 3: a study

A database says what its products are made of. A **study** asks a question of databases: which products, how much of each, scored with which methods, and in which scenarios. It is a folder next to the databases, with a `study.yaml`. It holds no processes.

The study below asks for one tote bag as Parts 1 and 2 wrote it, without the dye. So take the dye out again first: line `e06`, and `uses: [dyes]`. The `dyes` folder stays next to the tote, for a second product at the end of this part.

In [20]:
lines.write_text(
    lines.read_text().replace("e06,input,dyes:reactive-dye,0.01,kg,dyes:reactive-dye\n", "")
)
settings = database_folder / "database.yaml"
settings.write_text(settings.read_text().replace("uses: [dyes]\n", ""))
assert math.isclose(hw.open(database_folder).calc(bg, "tote:tote").score, 0.60927, rel_tol=1e-9)

## `study.yaml`: the question

`study.yaml` holds the goal and scope that ISO 14044 asks for:

- `name`, `goal` and `functional_unit`: text for people.
- `products`: what meets the functional unit, each a process id and an amount. `tote:tote` is found in the folder `tote` next to the study, by the start of its id.
- `boundary`: `cradle-to-gate`, `cradle-to-grave` or `gate-to-gate`.
- `methods`: the methods the study is scored with.
- `cut_off`: the criteria, and the largest share of inputs that may be left out in all. A database the study reaches may not allow more. The tote allows 0.03, so the study allows 0.03 too.

`hw.open` reads a study folder as it reads a database folder, and returns a `Study`. `study.calc(bg)` gives one `Result` per product, for its amount: here one tote bag, 0.60927 as in Part 1.

In [21]:
study_folder = databases / "tote-footprint"
write(
    "study.yaml",
    """\
name: tote-footprint
goal: Footprint of one tote bag, a teaching example
functional_unit: 1 tote bag
products:
  - {process: tote:tote, amount: 1}
boundary: cradle-to-gate
methods: [toy-climate]
cut_off: {criteria: [mass], max_cumulative: 0.03}
""",
    study_folder,
)
study = hw.open(study_folder)
print(study)
print(study.validate(bg))
results = study.calc(bg)
print(results)
assert math.isclose(results[0].score, 0.60927, rel_tol=1e-9)

Study(tote-footprint: 1 product, methods toy-climate)
Report(no problems)
(Result(tote:tote = 0.60927 kg CO2-eq, indicator climate-change, method toy-climate 1, 1 flow not characterised),)


`headwater schema FOLDER` writes JSON schemas of `database.yaml`, `study.yaml` and the scenario files into `FOLDER/.schemas`, which an editor can use to complete keys as you write them.

## `scenarios/`: named what-ifs

A scenario is a file `scenarios/<name>.yaml` of the study, with a description and a list of changes: to a line, or to a parameter. It is calculated with `study.calc(bg, scenario="<name>")`, and the files themselves do not change. A scenario reaches every database the study's products reach, and the background.

Two scenarios:

- `bigger-bag`: the bag area goes from 0.5 to 0.6 m², so 0.18 kg of fabric. By hand: 0.18 × 3.7086 + 0.05298 = 0.720528.
- `clean-grid`: the grid emits 0.3 kg CO₂ per kWh, **for everything**: the tote's electricity and the fabric's. By hand: electricity 0.3298, fabric 2.1192 + 3 × 0.3298 = 3.1086, tote 0.15 × 3.1086 + 0.1 × 0.3298 = 0.49927.

In [22]:
write(
    "scenarios/bigger-bag.yaml",
    """\
description: A bigger bag, 0.6 m2 of fabric instead of 0.5.
changes:
  - parameter: bag_area_m2
    value: 0.6
""",
    study_folder,
)
write(
    "scenarios/clean-grid.yaml",
    """\
description: The grid emits 0.3 kg CO2 per kWh instead of 0.5, for everything that buys it.
changes:
  - process: toy:electricity-production
    line: e02
    amount: 0.3
""",
    study_folder,
)
study = hw.open(study_folder)
print(study)
bigger = study.calc(bg, scenario="bigger-bag")[0]
cleaner = study.calc(bg, scenario="clean-grid")[0]
print(bigger)
print(cleaner)
assert math.isclose(bigger.score, 0.720528, rel_tol=1e-9)
assert math.isclose(cleaner.score, 0.49927, rel_tol=1e-9)

Study(tote-footprint: 1 product, methods toy-climate, scenarios bigger-bag clean-grid)
Result(tote:tote = 0.720528 kg CO2-eq, indicator climate-change, method toy-climate 1, scenario bigger-bag, 1 flow not characterised)
Result(tote:tote = 0.49927 kg CO2-eq, indicator climate-change, method toy-climate 1, scenario clean-grid, 1 flow not characterised)


Compare `clean-grid` with the override: both say the electricity emits 0.3. The override gave 0.58927, because only the tote's own 0.1 kWh changed. The scenario gives 0.49927, because the fabric's electricity changed too. An override is a fact about your supplier, so it is a file of the database. A scenario is a what-if about the world, so it is a file of the study. The background release is never edited by either.

## A second product, and a study in Python

A study can name several products, each with its amount, and scores each one for its amount. `hw.Study` makes a study in memory, from its products and methods. `databases` is the folder that holds the database folders: the study finds `tote` and `dyes` there.

Here: 2 tote bags and 0.5 kg of the reactive dye. By hand: 2 × 0.60927 = 1.21854, and 0.5 × 5.0596 = 2.5298.

In [23]:
batch = hw.Study(
    products={"tote:tote": 2, "dyes:reactive-dye": 0.5},
    methods=["toy-climate"],
    databases=databases,
)
print(batch)
tote_result, dye_result = batch.calc(bg)
print(tote_result)
print(dye_result)
assert math.isclose(tote_result.score, 1.21854, rel_tol=1e-9)
assert math.isclose(dye_result.score, 2.5298, rel_tol=1e-9)

Study(in memory: 2 products, methods toy-climate, not saved)
Result(2 × tote:tote = 1.21854 kg CO2-eq, indicator climate-change, method toy-climate 1, 1 flow not characterised)
Result(0.5 × dyes:reactive-dye = 2.5298 kg CO2-eq, indicator climate-change, method toy-climate 1)


`batch.save` writes it to a new folder next to its databases. `study.yaml` needs a goal and scope, so `save` asks for them: `goal`, `functional_unit`, `boundary` and `cut_off`. It writes to a copy first and reads it back, so nothing is written when the files would not read. Databases and studies then sit side by side in one folder.

In [24]:
saved = batch.save(
    databases / "tote-and-dye",
    goal="Two tote bags and the dye for one batch, a teaching example",
    functional_unit="2 tote bags and 0.5 kg of reactive dye",
    boundary="cradle-to-gate",
    cut_off={"criteria": ["mass"], "max_cumulative": 0.03},
)
print(saved)
show("study.yaml", databases / "tote-and-dye")
print(sorted(path.name for path in databases.iterdir()))
reread = hw.open(databases / "tote-and-dye").calc(bg)
assert math.isclose(reread[0].score, 1.21854, rel_tol=1e-9)
assert math.isclose(reread[1].score, 2.5298, rel_tol=1e-9)

Study(tote-and-dye: 2 products, methods toy-climate)
study.yaml:
name: tote-and-dye
goal: Two tote bags and the dye for one batch, a teaching example
functional_unit: 2 tote bags and 0.5 kg of reactive dye
products:
  - {process: tote:tote, amount: 2}
  - {process: dyes:reactive-dye, amount: 0.5}
boundary: cradle-to-gate
methods: [toy-climate]
cut_off: {criteria: [mass], max_cumulative: 0.03}

['dyes', 'tote', 'tote-and-dye', 'tote-footprint']


# Part 4: when something is wrong

headwater stops with a clear message instead of returning a wrong number. There are two places where a database can stop:

- `hw.open(folder)` reads the files. It stops if a file is missing or misnamed, a field is missing, or a value cannot be read.
- `db.calc(bg, "tote:tote")` and `db.validate(bg)` check the database against the background. They stop if a unit is unknown, a supplier does not exist or makes something else, or a flow has no factor.

Both raise `InvalidDataError`. It carries every problem found, not only the first. Each problem has a `code` that tools can match on.

The mistakes below are made on a fresh copy of the four files of Part 1. The helper `mistake` changes one piece of one file, prints the change as a diff, and puts the file back when its `with` block ends.

In [25]:
mistakes_folder = databases / "mistakes/tote"
write(
    "database.yaml",
    "name: tote\nbackground: toy-background@fixture\nmethods: [toy-climate]\n",
    mistakes_folder,
)
write("flows.csv", "id,name,kind,unit\ntote:tote-bag,tote bag,product,unit\n", mistakes_folder)
write("processes/tote.yaml", (database_folder / "processes/tote.yaml").read_text(), mistakes_folder)
write(
    "processes/tote.csv",
    """\
line,role,flow,amount,unit,supplier
e01,reference,tote:tote-bag,1,unit,
e02,input,toy:cotton-fabric,0.15,kg,toy:fabric-production
e03,input,toy:electricity,0.1,kWh,toy:electricity-production
""",
    mistakes_folder,
)
assert math.isclose(hw.open(mistakes_folder).calc(bg, "tote:tote").score, 0.60927, rel_tol=1e-9)


@contextmanager
def mistake(name, old, new):
    path = mistakes_folder / name
    original = path.read_text()
    assert original.count(old) == 1, f"{old!r} must appear once in {name}"
    changed = original.replace(old, new)
    diff = difflib.unified_diff(
        original.splitlines(), changed.splitlines(), "before", "after", lineterm="", n=0
    )
    print(name)
    print("\n".join(list(diff)[2:]), end="\n\n")
    path.write_text(changed)
    try:
        yield
    finally:
        path.write_text(original)

## A missing field in `database.yaml`

Remove `background`. The field is required, because a database must say which release it was written for. The message names the file and the field.

In [26]:
with mistake("database.yaml", "background: toy-background@fixture\n", ""):
    expect_error(mistakes_folder, "invalid-field", "hw.open")

database.yaml
@@ -2 +1,0 @@
-background: toy-background@fixture

Stopped in hw.open:
1 problem:
- database.yaml: background: Field required


## A process with only one of its two files

Rename `tote.yaml` to `tote.yml`, a common slip. Two problems come back: the `.yml` file is not a process file, and the `.csv` has no `.yaml` beside it. A misspelt file is never ignored.

In [27]:
good = mistakes_folder / "processes/tote.yaml"
slip = good.with_name("tote.yml")
good.rename(slip)
expect_error(mistakes_folder, "missing-file", "hw.open")
slip.rename(good)
assert good.exists()

Stopped in hw.open:
2 problems:
- processes/tote.yml: is not a process file. A process is a .csv file (its exchanges) with a .yaml file of the same name (its name, location and kind). [processes/tote.yml]
- processes/tote.csv: has no processes/tote.yaml beside it. A process needs both: a .csv file for its exchanges and a .yaml file for its name, location and kind. [processes/tote.csv]


## A role that does not exist

Write `buy` where `input` belongs. The message lists the roles that exist. `processes/tote.csv:3` is row 3 of that file, counting the header.

In [28]:
with mistake("processes/tote.csv", "e02,input,", "e02,buy,"):
    expect_error(mistakes_folder, "unknown-role", "hw.open")

processes/tote.csv
@@ -3 +3 @@
-e02,input,toy:cotton-fabric,0.15,kg,toy:fabric-production
+e02,buy,toy:cotton-fabric,0.15,kg,toy:fabric-production

Stopped in hw.open:
1 problem:
- tote:tote line e02: unknown role 'buy'. Roles: reference, input, own_use, waste_to_treatment, recyclable, avoided_product, emission, resource [processes/tote.csv:3]


## An amount that is not a number

Write `lots` for 0.15. Any text that is not a number is read as a formula, and `lots` is not a parameter.

In [29]:
with mistake("processes/tote.csv", "0.15", "lots"):
    expect_error(mistakes_folder, "bad-number", "hw.open")

processes/tote.csv
@@ -3 +3 @@
-e02,input,toy:cotton-fabric,0.15,kg,toy:fabric-production
+e02,input,toy:cotton-fabric,lots,kg,toy:fabric-production

Stopped in hw.open:
1 problem:
- tote:tote line e02: amount 'lots' is not a number. A formula can use the parameters of parameters.yaml, and this database has none. [processes/tote.csv:3]


## A negative amount

Write -0.15 for an input. You buy a positive amount; the role sets the direction. Only emissions and resources may be negative, because a few real datasets have negative ones.

In [30]:
with mistake("processes/tote.csv", "0.15", "-0.15"):
    expect_error(mistakes_folder, "negative-amount", "hw.open")

processes/tote.csv
@@ -3 +3 @@
-e02,input,toy:cotton-fabric,0.15,kg,toy:fabric-production
+e02,input,toy:cotton-fabric,-0.15,kg,toy:fabric-production

Stopped in hw.open:
1 problem:
- tote:tote line e02: amount -0.15 is negative, but role 'input' needs a positive amount. Only emission and resource may be negative. [processes/tote.csv:3]


That was the last mistake `hw.open` can see on its own. The next ones need the background, so they stop in `db.calc`.

## An unknown unit

Write `kgs` for `kg`. headwater converts between units of the same kind, so it must know the unit. It suggests the closest one.

In [31]:
with mistake("processes/tote.csv", ",kg,toy:fabric-production", ",kgs,toy:fabric-production"):
    expect_error(mistakes_folder, "unknown-unit", "db.calc")

processes/tote.csv
@@ -3 +3 @@
-e02,input,toy:cotton-fabric,0.15,kg,toy:fabric-production
+e02,input,toy:cotton-fabric,0.15,kgs,toy:fabric-production

Stopped in db.calc:
1 problem:
- tote:tote line e02: unknown unit 'kgs' (flow toy:cotton-fabric). Did you mean 'kg'?


## A supplier that does not exist

Mistype the supplier. It suggests the closest id.

In [32]:
with mistake("processes/tote.csv", "toy:fabric-production", "toy:fabric-productoin"):
    expect_error(mistakes_folder, "supplier-not-found", "db.calc")

processes/tote.csv
@@ -3 +3 @@
-e02,input,toy:cotton-fabric,0.15,kg,toy:fabric-production
+e02,input,toy:cotton-fabric,0.15,kg,toy:fabric-productoin

Stopped in db.calc:
1 problem:
- tote:tote line e02: supplier 'toy:fabric-productoin' does not exist (flow 'toy:cotton-fabric'). Did you mean 'toy:fabric-production'?


## A supplier that makes something else

Name the fabric process as the supplier of the electricity. The message says what the supplier makes.

In [33]:
with mistake("processes/tote.csv", "kWh,toy:electricity-production", "kWh,toy:fabric-production"):
    expect_error(mistakes_folder, "supplier-wrong-flow", "db.calc")

processes/tote.csv
@@ -4 +4 @@
-e03,input,toy:electricity,0.1,kWh,toy:electricity-production
+e03,input,toy:electricity,0.1,kWh,toy:fabric-production

Stopped in db.calc:
1 problem:
- tote:tote line e03: supplier 'toy:fabric-production' makes 'toy:cotton-fabric', not 'toy:electricity'


## A role that does not fit the flow

Make the fabric an `emission`. Emissions are elementary flows, things that cross the border to nature, and cotton fabric is a product. This one line breaks two rules, and both are listed.

In [34]:
with mistake("processes/tote.csv", "e02,input,", "e02,emission,"):
    expect_error(mistakes_folder, "role-flow-kind", "db.calc")

processes/tote.csv
@@ -3 +3 @@
-e02,input,toy:cotton-fabric,0.15,kg,toy:fabric-production
+e02,emission,toy:cotton-fabric,0.15,kg,toy:fabric-production

Stopped in db.calc:
2 problems:
- tote:tote line e02: role 'emission' needs an elementary flow, but 'toy:cotton-fabric' is a product flow
- tote:tote line e02: role 'emission' takes no supplier, but 'toy:fabric-production' is given


## Several problems at once

headwater does not stop at the first problem. Make the unit and the supplier both wrong, and both come back in one message, so you can fix them in one go.

In [35]:
with mistake("processes/tote.csv", ",kg,toy:fabric-production", ",kgs,toy:fabric-productoin"):
    expect_error(mistakes_folder, "unknown-unit", "db.calc")

processes/tote.csv
@@ -3 +3 @@
-e02,input,toy:cotton-fabric,0.15,kg,toy:fabric-production
+e02,input,toy:cotton-fabric,0.15,kgs,toy:fabric-productoin

Stopped in db.calc:
2 problems:
- tote:tote line e02: unknown unit 'kgs' (flow toy:cotton-fabric). Did you mean 'kg'?
- tote:tote line e02: supplier 'toy:fabric-productoin' does not exist (flow 'toy:cotton-fabric'). Did you mean 'toy:fabric-production'?


## A process the database does not have

Ask for `tote:bag` instead of `tote:tote`. `calc` scores one unit of what a process makes, so it needs the id of a process of the database. A wrong id is a wrong argument, not wrong data, so it is a `ValueError`. The message lists the processes the database has.

In [36]:
try:
    hw.open(mistakes_folder).calc(bg, "tote:bag")
except ValueError as error:
    print(error)
else:
    raise AssertionError("the process id should have been refused")

database tote has no process 'tote:bag'. Its processes: tote:tote


Three more mistakes were shown in Part 2: an input with no supplier (`supplier-missing`, fixed with a cut-off), cut-off rows with no `cut_off` in `database.yaml` (`cut-off-no-limit`), and a flow the method does not count (`no-factor`, fixed with `no_factor.csv`).

## What you did

- Wrote the four files every database has, and calculated it.
- Added each optional file: an override, parameters with a formula, a cut-off, a flow with no factor, and a database it uses. Each changed the footprint by a number you could check by hand, or recorded a decision with a reason.
- Wrote a study next to the database, with its goal and scope, its products and amounts, and two scenarios. Made one in Python with `hw.Study` and saved it.
- Read the message for each kind of mistake.

Next: [03-working-with-ecoinvent](03-working-with-ecoinvent.ipynb) does the same on a real background, ecoinvent 3.12. It needs an ecoinvent licence. Or [04-coming-from-brightway](04-coming-from-brightway.ipynb), which sets each task side by side with Brightway.